In [1]:
!pip install -q fastparquet
!pip install -q neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 5.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # оставляем только первую GPU

import torch
torch.multiprocessing.set_start_method('spawn', force=True)

In [3]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, NHITS

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [4]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

# данные о праздниках 
holidays = pd.read_excel('/kaggle/input/datasets/faibus/diploma/holidays.xlsx')
holidays['Holidays'] = pd.to_datetime(holidays['Holidays'], format='%Y.%m.%d')

# данные о погоде
weather = pd.read_csv('/kaggle/input/datasets/faibus/diploma/weather.csv')
weather['final_temp'] = (weather['temperature_ekb'] * 0.14) + (weather['temperature_msk'] * 0.22) + (weather['temperature_nov'] * 0.11) + (weather['temperature_ros'] * 0.16) + (weather['temperature_sam'] * 0.15) + (weather['temperature_spb'] * 0.09) + (weather['temperature_tve'] * 0.13)
weather = weather[['date', 'final_temp']]
weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m-%d')

# данные о днях промо
promo_days = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/promo_days.parquet', engine='fastparquet')

# данные о характеристиках SKU
sku_charasterictics = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/sku_features.parquet', engine='fastparquet')[['SKU_id', 'Category4', 'Brand', 'SupplierID']]

In [5]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'smooth' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(730449, 3)

In [6]:
# джойним характеристики SKU
real_demand = pd.merge(real_demand, sku_charasterictics, left_on = ['unique_id'], right_on = ['SKU_id'])

# создаём дни праздников (корректнее сказать "признак выходных")
real_demand['is_holiday'] = real_demand['ds'].isin(holidays['Holidays'])

# джойним промо-дни
real_demand = pd.merge(real_demand, promo_days, how = 'left', left_on = ['ds', 'unique_id'], right_on = ['date', 'SKU_id'])

# джойним погодные условия (температуру)
real_demand = pd.merge(real_demand, weather, how = 'left', left_on = ['ds'], right_on = ['date'])

# заполяем пропуски в промо-днях значением False
real_demand['is_promo'] = real_demand['is_promo'].fillna(False)

# оставляем только нужные колонки
real_demand = real_demand[['unique_id', 'ds', 'y', 'is_holiday', 'is_promo','final_temp', 'Category4', 'Brand', 'SupplierID']].copy()

# генерим временные фичи             
real_demand['month'] = real_demand['ds'].dt.month
real_demand['day'] = real_demand['ds'].dt.day
real_demand['dayofweek'] = real_demand['ds'].dt.dayofweek + 1 

# меняем тип данных bool -> int
real_demand['is_holiday'] = real_demand['is_holiday'].astype(int)
real_demand['is_promo'] = real_demand['is_promo'].astype(int)

real_demand.shape

/tmp/ipykernel_55/3064304324.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  real_demand['is_promo'] = real_demand['is_promo'].fillna(False)


(730449, 12)

### Применяем Label Encoding для категориальных колонок

In [7]:
# Кодируем в категории (обязательно для NeuralForecast)
static_cols = ['Category4', 'Brand', 'SupplierID']
for col in static_cols:
    real_demand[col] = real_demand[col].astype('category').cat.codes

real_demand.shape

(730449, 12)

### Подготовка рядов к обучению

In [8]:
### Фильтрация рядов, достаточно длинных для обучения

real_demand_filtered = filter_long_series(
    real_demand,
    horizon = 14,
    n_val_windows = 1,
    n_test_windows = 3,
    min_train_points = 35,
    id_col="unique_id",
)



### Разделение на train, val, test

train_df, val_df, test_windows_df = split_train_val_test(
    real_demand_filtered,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    id_col="unique_id",
    ds_col="ds",
)

### Обучение

In [9]:
horizon = 14

# Задаем экзогенные переменные
exog_cols = ['is_holiday', 'is_promo', 'final_temp', 'month', 'day', 'dayofweek']
static_cols = ['Category4', 'Brand', 'SupplierID']

# Общие параметры для всех моделей
trainer_params = {
    'accelerator': 'gpu',
    'devices': 1,
    'strategy': 'auto' 
}

# Задаем модели
def make_models():
    common = dict(
        h=horizon,
        scaler_type='robust',
        futr_exog_list=exog_cols,
        stat_exog_list=static_cols,
        **trainer_params
    )
    return [
        # === LSTM ===
        LSTM(**common, input_size=21, max_steps=1000, alias='LSTM'),

        # === GRU ===
        GRU(**common, input_size=21, max_steps=1000, alias='GRU'),

        # === NHITS ===
        NHITS(**common, input_size=35, max_steps=1000,
              n_freq_downsample=[7, 1, 1],
              dropout_prob_theta=0.1,
              alias='NHITS'),
    ]


# Обучение

# fit только на train
static_train = train_df[['unique_id'] + static_cols].drop_duplicates('unique_id')
nf = NeuralForecast(models=make_models(), freq='D')
nf.fit(df=train_df, static_df=static_train)


def predict_one_window_frozen(nf, history_df, target_window_df, horizon=14):
    
    # Берем только full ids в текущем 14-дневном окне
    target_counts = target_window_df.groupby('unique_id')['ds'].nunique()
    full_target_ids = target_counts[target_counts == horizon].index

    hist_ids = pd.Index(history_df['unique_id'].unique())
    eligible_ids = hist_ids.intersection(full_target_ids)

    hist_part = history_df[history_df['unique_id'].isin(eligible_ids)].copy()
    target_part = target_window_df[target_window_df['unique_id'].isin(eligible_ids)].copy()

    static_part = hist_part[['unique_id'] + static_cols].drop_duplicates('unique_id')

    # expected future grid для этого history (без re-fit)
    expected_futr = nf.make_future_dataframe(df=hist_part)

    exog_source = (
        target_part[['unique_id', 'ds'] + exog_cols]
        .drop_duplicates(['unique_id', 'ds'])
    )
    futr_df = expected_futr.merge(exog_source, on=['unique_id', 'ds'], how='left')


    # predict с переданным history, но без обучения
    preds = nf.predict(df=hist_part, static_df=static_part, futr_df=futr_df).reset_index()

    if 'index' in preds.columns:
        preds = preds.drop(columns=['index'])

    out = target_part[['unique_id', 'ds', 'y']].merge(
        preds, on=['unique_id', 'ds'], how='inner'
    )
    return out

Seed set to 1
Seed set to 1
Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-27 16:13:01.130563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777306381.520293      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777306381.623570      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777306382.666359      55 computation_placer.cc:177] computation placer a

┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ hist_encoder │ LSTM          │  203 K │ train │     0 │
│ 4 │ mlp_decoder  │ MLP           │ 17.4 K │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 221 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 221 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  2.8 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11                                                                         
Modules in train mode: 43                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=1000` reached.


### Прогнозы

In [10]:
# Обертка
def predict_fn(history_df, target_window_df, horizon):
    return predict_one_window_frozen(
        nf=nf,
        history_df=history_df,
        target_window_df=target_window_df,
        horizon=horizon,
    )

# Сбор предсказаний
val_pred, test_pred = evaluate_frozen_windows(
    predict_one_window_fn=predict_fn,
    train_df=train_df,
    val_df=val_df,
    test_windows_df=test_windows_df,
    horizon=horizon,
    ds_col="ds",
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

### Расчет метрик

In [11]:
# -------- VAL metrics --------
val_model_cols = get_model_columns(val_pred, reserved_columns = ("unique_id", "ds", "cutoff", "y", "index", "test_window"))
val_metrics_per_window = compute_metrics_per_window(
    cv_results=val_pred,
    model_columns=val_model_cols,
    metrics_dict=DEFAULT_METRICS
)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)


# -------- TEST metrics --------
test_model_cols = get_model_columns(test_pred, reserved_columns = ("unique_id", "ds", "cutoff", "y", "index", "test_window"))
test_metrics_per_window = compute_metrics_per_window(
    cv_results=test_pred,
    model_columns=test_model_cols,
    metrics_dict=DEFAULT_METRICS
)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)
print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
GRU,5.121205,11.469247,56.488950,40.890149
LSTM,5.188810,11.474450,56.561291,41.429939
NHITS,4.744928,10.642480,52.592331,37.885774



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
GRU,4.726976,10.006586,55.960299,38.012991
LSTM,4.786219,10.054212,56.162751,38.491547
NHITS,4.493841,9.113776,54.522114,36.136757


In [12]:
test_pred.to_csv("neural_forecast_smooth_test_pred.csv")